# MVP + выбранный v21 — v23 / Run All

Проверяем объединение лучших конкретных сохранённых моделей. **Без нового обучения и повторного encoder inference**: используем готовые эмбеддинги завершённого v22.

**Restart Kernel → Run All** в прежнем исследовательском `.venv`. Интернет, GPU и другое устройство не нужны. Общего ограничения времени нет.

Сетка заранее фиксирована: доля MVP 25/50/75% × λ0.50/0.65 плюс два чистых контроля. Выбор по mAP@10 на calibration; при равенстве C, затем исходный порядок сетки. Порог каждого варианта подбирается только на calibration. На validation — только зафиксированный выбор и контроли.

Это development-поиск на уже наблюдавшихся данных, не независимый hidden-test результат. MVP, исходные bbox/validation, веса и старые runs не меняются.


In [1]:
from pathlib import Path
import os, sys, subprocess
candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/mvp_v21_fusion.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из исследовательского Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import Markdown, display
from training import mvp_v21_fusion as experiment
SOURCE_RUN = 'comparison_v1'
RUN_NAME = 'fusion_v1'
ALLOW_OUTER_EVALUATION = True  # Согласованный development-прогон после выбора на calibration.
print('Repository:', REPO, '\nKernel:', sys.executable)
print('Run:', RUN_NAME, '| CPU: только сохранённые массивы | обучение: нет | encoder inference: нет')


Repository: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK 
Kernel: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python
Run: fusion_v1 | CPU: только сохранённые массивы | обучение: нет | encoder inference: нет


## 1. Быстрые проверки
Геометрия смешивания 512+1536, выбор только на calibration, независимость query, экспорт и возобновление. Тесты используют синтетические данные.


In [2]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_mvp_v21_fusion.py', 'tests/test_final_model_comparison.py'], check=True)


..................................                                       [100%]
34 passed in 3.33s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_mvp_v21_fusion.py', 'tests/test_final_model_comparison.py'], returncode=0)

## 2. Проверка кэша v22 и фиксация сетки
Кэши не пересчитываются. Отсутствующий или повреждённый источник останавливает работу. Составы и сетка сохраняются до вычисления новых метрик.


In [3]:
context = experiment.prepare(RUN_NAME, source_run=SOURCE_RUN)
print('Результаты:', context['output'])
for spec in context['manifest']['fusion_plan']['systems']:
    print(spec['name'], '| доля MVP:', spec['mvp_weight'], '| lambda:', spec['lambda'], '| кандидат:', spec['candidate_policy'])


Результаты: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_23_mvp_v21_fusion/runs/fusion_v1
MVP_control | доля MVP: 1.0 | lambda: 0.5 | кандидат: ranking_top1
V21_control | доля MVP: 0.0 | lambda: 0.65 | кандидат: raw_top1
mix_mvp75_lambda50 | доля MVP: 0.75 | lambda: 0.5 | кандидат: raw_top1
mix_mvp75_lambda65 | доля MVP: 0.75 | lambda: 0.65 | кандидат: raw_top1
mix_mvp50_lambda50 | доля MVP: 0.5 | lambda: 0.5 | кандидат: raw_top1
mix_mvp50_lambda65 | доля MVP: 0.5 | lambda: 0.65 | кандидат: raw_top1
mix_mvp25_lambda50 | доля MVP: 0.25 | lambda: 0.5 | кандидат: raw_top1
mix_mvp25_lambda65 | доля MVP: 0.25 | lambda: 0.65 | кандидат: raw_top1


## 3. Calibration → frozen_selection → validation → отчёт
Показываются этап, конфигурация, метрики calibration и время. Здесь нет эпох: модели уже обучены. После ошибки повтори Run All с тем же RUN_NAME. Контроль может выиграть — смесь не назначается победителем принудительно.


In [4]:
result = experiment.run(context, allow_outer=ALLOW_OUTER_EVALUATION)



STAGE 1/3: eight calibration configurations, saved features only
Loaded verified calibration embeddings: 307 query + 895 gallery; no inference
[START] calibrate_MVP_control | elapsed 0.00 h
  MVP_control: calibration mAP=0.787185, C=0.724868
[DONE] calibrate_MVP_control | 0.1 min
[START] calibrate_V21_control | elapsed 0.01 h
  V21_control: calibration mAP=0.822982, C=0.772850
[DONE] calibrate_V21_control | 0.1 min
[START] calibrate_mix_mvp75_lambda50 | elapsed 0.01 h
  mix_mvp75_lambda50: calibration mAP=0.807945, C=0.766300
[DONE] calibrate_mix_mvp75_lambda50 | 0.1 min
[START] calibrate_mix_mvp75_lambda65 | elapsed 0.01 h
  mix_mvp75_lambda65: calibration mAP=0.803843, C=0.766300
[DONE] calibrate_mix_mvp75_lambda65 | 0.1 min
[START] calibrate_mix_mvp50_lambda50 | elapsed 0.01 h
  mix_mvp50_lambda50: calibration mAP=0.812783, C=0.770267
[DONE] calibrate_mix_mvp50_lambda50 | 0.1 min
[START] calibrate_mix_mvp50_lambda65 | elapsed 0.01 h
  mix_mvp50_lambda65: calibration mAP=0.816556, C

In [5]:
display(Markdown((context['output'] / 'REPORT.md').read_text()))
print('Полные результаты:', context['output'] / 'results.json')
print('Выбор до validation:', context['output'] / 'frozen_selection.json')
assert result['status'] == 'complete' and result['protected_unchanged']
assert result['optimizer_updates'] == 0 and result['encoder_forwards'] == 0 and not result['promoted']


# v23 — MVP + v21 на сохранённых эмбеддингах

Статус: **complete**.
Нового обучения и encoder inference нет. Calibration/validation уже наблюдались: это development-поиск, не независимый тест.

## Calibration: вся заранее фиксированная сетка

Выбор по mAP@10; при равенстве C, затем исходный порядок сетки. Каждый порог — по calibration C.

| Система | mAP@10 | C | Порог |
|---|---:|---:|---:|
| MVP_control | 0.787185 | 0.724868 | 0.59487545 |
| V21_control | 0.822982 | 0.772850 | 0.52370554 |
| mix_mvp75_lambda50 | 0.807945 | 0.766300 | 0.56551540 |
| mix_mvp75_lambda65 | 0.803843 | 0.766300 | 0.56551540 |
| mix_mvp50_lambda50 | 0.812783 | 0.770267 | 0.55648959 |
| mix_mvp50_lambda65 | 0.816556 | 0.770267 | 0.55648959 |
| mix_mvp25_lambda50 | 0.820359 | 0.770972 | 0.52914703 |
| mix_mvp25_lambda65 | 0.821575 | 0.770972 | 0.52914703 |

Зафиксированный выбор: **V21_control**. После validation выбор не меняется.

## Validation: только выбор и контроли

| Система | mAP@10 | Rank-1 | F1 | TNR | C | 45×mAP+10×C | Принято / отказ |
|---|---:|---:|---:|---:|---:|---:|---:|
| MVP_control | 0.814689 | 0.801619 | 0.728606 | 0.790323 | 0.747121 | 44.1322 | 187 / 122 |
| V21_control | 0.810805 | 0.805668 | 0.800000 | 0.677419 | 0.763226 | 44.1185 | 225 / 84 |

Выбор против MVP_control: ΔmAP -0.388 п.п.; ΔC +1.610 п.п.; AP лучше/хуже/равно: 34/34/179.

Выбор против V21_control: ΔmAP +0.000 п.п.; ΔC +0.000 п.п.; AP лучше/хуже/равно: 0/0/247.

## Ограничения

- Смеси имеют 2048 измерений; доля v21 делится поровну между его тремя encoder. Контроли сохраняют исходные 512/1536 измерений и политики кандидатов.
- Выбор может остаться чистым MVP или v21. Нет обязательного выигрыша смеси и нет gate по среднему seed.
- Контроли обязаны точно воспроизвести v22: исходные threshold, метрики и per-query решения. Допуски не расширяются.
- Поиск проводится на calibration, потому что full-train MVP видел внутренние primary ID. Старые внутренние draws не подходят для честного сравнения этой смеси.
- 45×mAP+10×C — только два компонента из 55 возможных баллов; это не полный конкурсный балл и не критерий выбора вместо mAP.
- submission.csv содержит десять ID при любом отказе; кандидаты отсутствуют при отказе. Junk/top-10 считаются исходным evaluator без изменения меток.
- embeddings.npy содержит реальные float32-векторы в порядке query, затем gallery. Это validation-экспорты, не test submission и не готовый deployment bundle.
- Общего ограничения времени нет. Cached-задачи не являются новым измерением скорости. Ни GPU, ни сеть не нужны.
- Исходные файлы сохранены: True. MVP не меняется; promoted=False.
- Время текущего запуска: 1.3 мин.


Полные результаты: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_23_mvp_v21_fusion/runs/fusion_v1/results.json
Выбор до validation: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_23_mvp_v21_fusion/runs/fusion_v1/frozen_selection.json
